# Урок 9. Проверяем и очищаем данные осмысленно

**Зачем:** ошибка в данных может изменить ответ. Неизвестный возраст — не ноль лет, общий билет — не один и тот же пассажир. Сначала выясняем смысл значения, затем выбираем действие.

Сегодня сначала увидим и исправим ошибки в отдельной таблице из **10 строк**, затем применим проверки к своему Titanic. В основном Titanic сохраняем исторические сведения и добавляем понятные отметки. Удаление строк исследуем на отдельных копиях, чтобы увидеть, как меняется ответ.

**Что сдаём:** `homework/lesson-09/solution.ipynb` и `homework/lesson-09/data/titanic_clean.csv`. Собственные объяснения пишем в Markdown ноутбука. Отдельные отчёты качества, manifest и README для этого урока не требуются. Срок объявляет преподаватель.

P1–P4 — практика занятия; H1–H3 — домашняя часть в том же ноутбуке. Практику сохраняйте по команде преподавателя. Отсутствовавший выполняет обе части для ДЗ; отдельный зачёт практики занятия зависит от посещения.


## Как читать столбцы и команды

Английские имена сохраняем: это точные имена полей, которые использует Python.

| Имя | Значение |
|---|---|
| PassengerId | Номер записи пассажира |
| Survived | Исход: 1 — выжил, 0 — не выжил |
| Pclass | Класс билета: 1, 2 или 3 |
| Name / Sex | Имя / пол; значения Sex: female, male |
| Age | Возраст в годах; пустое значение означает «неизвестно» |
| SibSp / Parch | Число братьев, сестёр, супругов / родителей и детей на борту |
| Ticket / Fare | Билет / стоимость проезда |
| Cabin / Embarked | Каюта / код порта посадки: S, C, Q |
| Family_size / Is_alone | Размер семьи с учётом пассажира / едет один |
| Age_group | Учебная возрастная группа из урока 8 |
| Age_known | Известен ли возраст: True или False |
| Embarked_for_analysis | Подпись порта для анализа, включая «Неизвестно» |

`df` — имя таблицы, `df["Age"]` — один столбец, `df[["Age", "Fare"]]` — два столбца. `display(...)` показывает таблицу, `print(...)` печатает пояснение. `NaN` и `<NA>` означают отсутствие значения, а не ноль. `True` — «да», `False` — «нет».

`.loc[условие, "Age"]` выбирает подходящие строки и столбец Age; `.loc[условие]` показывает все столбцы выбранных строк.

Строка `name = выражение` сохраняет результат справа под именем слева. Комментарий после `#` не выполняется. В ячейках практики замените `pass` своим кодом; ниже напишите собственный вывод. Студенческий шаблон может выполниться без исключений, даже когда задания ещё не решены.


## Подготовка: открыть исходник без проверки хеша

1. Распакуйте архив урока. В Colab откройте ноутбук и слева выберите **Файлы → Загрузить**. Загрузите `titanic.csv` из архива.
2. Если исходник берёте с GitHub, скачайте сам CSV через **Raw / Download raw file**. Сохранённая HTML-страница просмотра не является таблицей.
3. При наличии загрузите свой `titanic_lesson08.csv`. Без него можно заниматься на восстановленном учебном этапе, но прежнее ДЗ от этого не засчитывается.
4. Выполните готовую ячейку ниже. Она напечатает выбранный путь и первые три строки. Ожидается 891 пассажир и 342 выживших.

Технический блок писать по памяти не нужно. Он принимает одинаковые CSV с разными переводами строк и служебной меткой UTF-8. Если рядом лежит неподходящий файл, проверяет другие найденные варианты. При нескольких копиях смотрите напечатанный путь; нужный файл можно явно указать в `DATA_FILE`. Загрузка `titanic (1).csv` тоже поддерживается.

В Colab загруженные файлы временные: после сброса среды загрузите их снова. Сообщение «файл не найден» означает, что нужно загрузить файл; «не найдены столбцы» — проверить CSV и способ скачивания. Сохранность важных сведений проверяется по содержимому таблицы, а не по хешу файла.


In [1]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display

# Обычно менять эту строку не нужно. Здесь можно явно указать путь к своему исходнику.
DATA_FILE = None
SOURCE_COLUMNS = ["PassengerId", "Survived", "Pclass", "Name", "Sex", "Age",
                  "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked"]

def read_source(path):
    # utf-8-sig читает и обычный UTF-8, и UTF-8 со служебной меткой в начале.
    table = pd.read_csv(path, encoding="utf-8-sig")
    missing_columns = set(SOURCE_COLUMNS) - set(table.columns)
    if missing_columns:
        raise ValueError("Не найдены столбцы: " + ", ".join(sorted(missing_columns)))
    if len(table) != 891:
        raise ValueError(f"Ожидалось 891 пассажир, получено {len(table)}. Нужен выданный titanic.csv.")
    if not table["PassengerId"].is_unique or set(table["PassengerId"]) != set(range(1, 892)):
        raise ValueError("Номера пассажиров отличаются от выданного исходника или повторяются.")
    if not table["Survived"].isin([0, 1]).all() or table["Survived"].sum() != 342:
        raise ValueError("Значения Survived отличаются от ожидаемых. Загрузите исходный CSV.")
    return table[SOURCE_COLUMNS].copy()

search_dirs = [Path.cwd(), Path.cwd() / "data"]
for base in [Path.cwd(), *Path.cwd().parents][:6]:
    search_dirs.extend([base / "07_Данные" / "lesson-09",
                        base / "01_Course" / "Python для ИИ" / "07_Данные" / "lesson-09"])
search_dirs.append(Path("/content/drive/MyDrive/Colab Notebooks"))
candidates = [Path(DATA_FILE)] if DATA_FILE else []
if not DATA_FILE:
    for folder in search_dirs:
        candidates.append(folder / "titanic.csv")
        if folder.is_dir():
            candidates.extend(sorted(p for p in folder.glob("titanic*.csv")
                                     if re.fullmatch(r"titanic\s*\(\d+\)\.csv", p.name)))
seen = set()
raw = None
errors = []
for candidate in candidates:
    if not candidate.is_file() or candidate.resolve() in seen:
        continue
    seen.add(candidate.resolve())
    try:
        raw = read_source(candidate)
        raw_path = candidate
        break
    except (ValueError, OSError, UnicodeError, pd.errors.ParserError) as error:
        errors.append(f"{candidate}: {error}")
if raw is None:
    detail = "\n".join(errors) if errors else "Файл titanic.csv не найден."
    raise ValueError(detail + "\nВ Colab: Файлы → Загрузить → выберите titanic.csv из архива урока. "
                     "На GitHub скачивайте Raw / Download raw file, а не страницу просмотра. "
                     "При другом имени укажите точный путь в DATA_FILE.")
if errors:
    print("Пропущены неподходящие файлы:")
    for error in errors:
        print(error)
print("Открыт исходник:", raw_path)

def restore_features(table):
    result = table.copy()
    result["Family_size"] = result["SibSp"] + result["Parch"] + 1
    result["Is_alone"] = result["Family_size"] == 1
    result["Age_group"] = "Неизвестный"
    result.loc[result["Age"] < 18, "Age_group"] = "Младше18"
    result.loc[result["Age"] >= 18, "Age_group"] = "От18"
    return result

df = restore_features(raw)
previous_paths = [Path("data/titanic_lesson08.csv"), Path("titanic_lesson08.csv"),
                  Path("../lesson-08/data/titanic_lesson08.csv")]
previous_path = next((p for p in previous_paths if p.is_file()), None)
if previous_path:
    try:
        previous = pd.read_csv(previous_path, encoding="utf-8-sig")
        pd.testing.assert_frame_equal(
            previous[SOURCE_COLUMNS].sort_values("PassengerId").reset_index(drop=True),
            raw.sort_values("PassengerId").reset_index(drop=True), check_dtype=False)
        expected = restore_features(raw).set_index("PassengerId")
        features = ["Family_size", "Is_alone", "Age_group"]
        pd.testing.assert_frame_equal(previous.set_index("PassengerId")[features].sort_index(),
                                      expected[features].sort_index(), check_dtype=False)
        df = previous.copy()
        print("Продолжаем личный этап:", previous_path)
    except (KeyError, ValueError, AssertionError, OSError, UnicodeError, pd.errors.ParserError) as error:
        print("Личный предыдущий файл требует исправления. Для занятия восстановлены исходник и признаки.")
        print("Причина:", str(error).split("\n")[0])
        print("Это не является сдачей или зачётом предыдущей работы.")
else:
    print("Личный файл урока 8 не найден. Для занятия признаки восстановлены из исходника.")
    print("Это не является сдачей или зачётом предыдущей работы.")
stage_result = None
print("Пассажиров:", len(df), "Выжили:", int(df["Survived"].sum()))
display(df.head(3))


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_size,Is_alone,Age_group
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,False,От18
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C,2,False,От18
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,True,От18


Открыт исходник: /workspace/scratch/3a928a655d6e/lesson09-update/checks/student/titanic.csv
Личный файл урока 8 не найден. Для занятия признаки восстановлены из исходника.
Это не является сдачей или зачётом предыдущей работы.
Пассажиров: 891 Выжили: 342


## Учебная таблица: все 10 строк видны на экране

Это **вымышленные записи**, а не новые пассажиры Titanic. В них специально оставлены ошибки. Посмотрите на всю таблицу и до запуска проверок найдите хотя бы три подозрительных места. Ничего не исправляйте догадкой.

Здесь используются уже знакомые имена PassengerId, Age, Sex, Pclass, Fare и Embarked. Некоторые числа пока записаны текстом. Номер строки слева — индекс pandas; PassengerId — значение отдельного столбца.


In [2]:
toy = pd.DataFrame({
    "PassengerId": [101, 102, 103, 104, 105, 106, 106, 107, 108, 107],
    "Age": ["22", None, "-2", "тридцать", "250", "18", "19", " 40 ", "30", " 40 "],
    "Sex": [" Female ", "male", "female", " male", "FEMALE", "male ", "male", "female", "male", "female"],
    "Pclass": [1, 2, 4, 3, 2, 3, 3, 1, 3, 1],
    "Fare": ["80", "0", "15", "-5", "25", "12.5", "12.5", "100", "7.25", "100"],
    "Embarked": ["S", "C", "", " Q ", " ", "S", "S", "C", None, "C"]
})
display(toy)
print("Строк:", len(toy))


,PassengerId,Age,Sex,Pclass,Fare,Embarked
0,101,22,Female,1,80,S
1,102,None,male,2,0,C
2,103,-2,female,4,15,
3,104,тридцать,male,3,-5,Q
4,105,250,FEMALE,2,25,
5,106,18,male,3,12.5,S
6,106,19,male,3,12.5,S
7,107,40,female,1,100,C
8,108,30,male,3,7.25,None
9,107,40,female,1,100,C


Строк: 10


### Пример 1. Методы выполняются друг за другом

Хотим сравнивать категории Sex. Строки `" Female "`, `"FEMALE"` и `"female"` имеют одинаковый смысл, но для компьютера различаются.

1. `.str` даёт доступ к операциям с текстом в столбце.
2. `.str.strip()` убирает пробелы по краям: `" Female "` → `"Female"`.
3. `.str.lower()` приводит буквы к нижнему регистру: `"Female"` → `"female"`.

Сначала покажем каждый промежуточный результат. Затем соединим операции в одну цепочку: результат слева становится входом следующего метода. Порядок читаем слева направо. Нужное изменение сохраняем присваиванием. `toy.copy()` создаёт учебную рабочую копию, оставляя исходную toy прежней. В показанной таблице before — исходник, after_strip — результат удаления пробелов, after_lower — результат приведения регистра.


In [3]:
sex_before = toy["Sex"]
sex_without_spaces = sex_before.str.strip()
sex_lowercase = sex_without_spaces.str.lower()
display(pd.DataFrame({"before": sex_before, "after_strip": sex_without_spaces, "after_lower": sex_lowercase}))

# Та же последовательность одной строкой. Проверяем, что ответы совпали.
sex_chain = toy["Sex"].str.strip().str.lower()
print("Пошаговый результат совпал с цепочкой:", sex_chain.equals(sex_lowercase))
toy_clean = toy.copy()
toy_clean["Sex"] = sex_chain


,before,after_strip,after_lower
0,Female,Female,female
1,male,male,male
2,female,female,female
3,male,male,male
4,FEMALE,FEMALE,female
5,male,male,male
6,male,male,male
7,female,female,female
8,male,male,male
9,female,female,female


Пошаговый результат совпал с цепочкой: True


### Пустая клетка, пустая строка и пробелы

`.isna()` находит пропуски, но строка `""` и строка `" "` первоначально являются текстом. Поэтому для Embarked сначала уберём пробелы `.str.strip()`, приведём коды к верхнему регистру `.str.upper()`, затем заменим пустую строку настоящим пропуском `.replace("", pd.NA)`.

Мы не узнаём неизвестный порт. Мы приводим разные записи отсутствия сведений к одному понятному виду. У настоящего нуля Fare другой смысл — его пропуском не считаем.


In [4]:
print("Пропусков Embarked до обработки:", toy["Embarked"].isna().sum())
toy_clean["Embarked"] = toy["Embarked"].str.strip().str.upper().replace("", pd.NA)
display(pd.DataFrame({"before": toy["Embarked"], "after": toy_clean["Embarked"]}))
print("Пропусков после учёта пустых строк и пробелов:", toy_clean["Embarked"].isna().sum())


,before,after
0,S,S
1,C,C
2,,<NA>
3,Q,Q
4,,<NA>
5,S,S
6,S,S
7,C,C
8,None,None
9,C,C


Пропусков Embarked до обработки: 1
Пропусков после учёта пустых строк и пробелов: 3


### Число должно быть числом

Текст `"22"` похож на число, но тип у него другой. `pd.to_numeric(..., errors="coerce")` пытается получить число: `"22"` станет 22, а `"тридцать"` превратится в пропуск. Мы **не угадываем** возраст по слову: только отмечаем неудачное преобразование.

Сохраним прежний столбец в отдельной переменной, чтобы показать, какие известные записи не удалось преобразовать. Число 250 преобразуется успешно — это ещё не означает, что возраст допустим. Проверка типа и проверка смысла различаются. `.notna()` отмечает известные исходные записи, `.isna()` — пропуски после преобразования. В условии ниже `&` означает «и»: исходная запись была непустой и результат стал пропуском. Каждое условие даёт True/False.


In [5]:
age_before = toy_clean["Age"].copy()
toy_clean["Age"] = pd.to_numeric(age_before, errors="coerce")
toy_clean["Fare"] = pd.to_numeric(toy_clean["Fare"], errors="coerce")
conversion_failed = age_before.notna() & toy_clean["Age"].isna()
display(pd.DataFrame({"before": age_before, "after": toy_clean["Age"], "conversion_failed": conversion_failed}))
print("Непустых записей Age, которые не стали числом:", conversion_failed.sum())
print("Тип Age после преобразования:", toy_clean["Age"].dtype)


,before,after,conversion_failed
0,22,22.0,False
1,None,NaN,False
2,-2,-2.0,False
3,тридцать,NaN,True
4,250,250.0,False
5,18,18.0,False
6,19,19.0,False
7,40,40.0,False
8,30,30.0,False
9,40,40.0,False


Непустых записей Age, которые не стали числом: 1
Тип Age после преобразования: float64


### Допустимые значения и доказанные ошибки

Для вымышленной таблицы договоримся: возраст от 0 до 120 лет, стоимость неотрицательна, Pclass входит в `[1, 2, 3]`. `|` означает «или», `&` — «и»; каждое сравнение берём в скобки. `.isin(...)` проверяет принадлежность списку. `~` меняет «да» на «нет» и наоборот.

Сначала покажем найденные строки, затем заменим недопустимые значения пропусками **в учебной копии**. Верные числа нам неизвестны. На реальных данных пределы согласуют с предметной областью, а подозрительные значения расследуют. Fare=0 сам по себе ошибкой не является.


In [6]:
bad_age = (toy_clean["Age"] < 0) | (toy_clean["Age"] > 120)
bad_fare = toy_clean["Fare"] < 0
bad_class = ~toy_clean["Pclass"].isin([1, 2, 3])
display(toy_clean.loc[bad_age | bad_fare | bad_class])
print("Недопустимый Age:", bad_age.sum(), "Fare:", bad_fare.sum(), "Pclass:", bad_class.sum())
toy_clean.loc[bad_age, "Age"] = pd.NA
toy_clean.loc[bad_fare, "Fare"] = pd.NA
toy_clean.loc[bad_class, "Pclass"] = pd.NA
display(toy_clean)


,PassengerId,Age,Sex,Pclass,Fare,Embarked
2,103,-2.0,female,4,15.0,<NA>
3,104,NaN,male,3,-5.0,Q
4,105,250.0,female,2,25.0,<NA>


,PassengerId,Age,Sex,Pclass,Fare,Embarked
0,101,22.0,female,1.0,80.00,S
1,102,NaN,male,2.0,0.00,C
2,103,NaN,female,NaN,15.00,<NA>
3,104,NaN,male,3.0,NaN,Q
4,105,NaN,female,2.0,25.00,<NA>
5,106,18.0,male,3.0,12.50,S
6,106,19.0,male,3.0,12.50,S
7,107,40.0,female,1.0,100.00,C
8,108,30.0,male,3.0,7.25,None
9,107,40.0,female,1.0,100.00,C


Недопустимый Age: 2 Fare: 1 Pclass: 1


### Повтор всей строки и конфликт номера

`.duplicated()` отмечает повторные вхождения после первого. `.duplicated(keep=False)` отмечает **всех** участников повтора. В этой вымышленной таблице мы знаем, что полностью одинаковая строка попала при повторной загрузке, поэтому уберём её `.drop_duplicates()`.

У двух оставшихся записей PassengerId=106 разный Age. Нельзя просто оставить первую и выдумать, какая верна. Покажем конфликт отдельно. Для дальнейшего учебного расчёта временно выделим записи с однозначными номерами, явно посчитаем исключённые строки. В основном Titanic пассажиров по повторяющемуся Ticket не удаляем.


In [7]:
print("Полных повторных строк до исправлений:", toy.duplicated().sum())
print("Повторных вхождений PassengerId до исправлений:", toy["PassengerId"].duplicated().sum())
toy_clean = toy_clean.drop_duplicates().copy()
conflict_mask = toy_clean["PassengerId"].duplicated(keep=False)
toy_conflicts = toy_clean.loc[conflict_mask].copy()
toy_ready = toy_clean.loc[~conflict_mask].copy()
display(toy_conflicts)
print("Исходных строк:", len(toy), "После удаления полного дубля:", len(toy_clean))
print("Отложены до уточнения:", len(toy_conflicts), "Для расчёта осталось:", len(toy_ready))
print("В исходной таблице по-прежнему:", len(toy))


,PassengerId,Age,Sex,Pclass,Fare,Embarked
5,106,18.0,male,3.0,12.5,S
6,106,19.0,male,3.0,12.5,S


Полных повторных строк до исправлений: 1
Повторных вхождений PassengerId до исправлений: 2
Исходных строк: 10 После удаления полного дубля: 9
Отложены до уточнения: 2 Для расчёта осталось: 7
В исходной таблице по-прежнему: 10


### Пример 2. Цепочка может менять вид результата

Посчитаем пропуски на вымышленной таблице после исправлений:

1. Таблица → `.isna()` → таблица ответов True/False.
2. Эта таблица → `.sum()` → по одному количеству на столбец. True считается за 1.
3. Эти количества → `.sort_values(ascending=False)` → список от большего количества к меньшему.

Это ещё одна последовательность методов. `.isna().sum()` не меняет исходные данные: создаёт результат проверки. Не любую функцию можно соединить с любой: следующий метод должен подходить тому виду результата, который получился.

В контроле ниже `.dropna()` временно исключает неизвестные значения только из проверки; `.between(0, 120)` проверяет диапазон включительно; `.all()` отвечает, все ли проверки истинны. Исходные пропуски этими проверками не заполняются.


In [8]:
missing_flags = toy_ready.isna()
display(missing_flags)
missing_counts = missing_flags.sum()
display(missing_counts)
missing_sorted = missing_counts.sort_values(ascending=False)
display(missing_sorted)
print("Цепочка дала тот же ответ:", toy_ready.isna().sum().sort_values(ascending=False).equals(missing_sorted))

# Контроль после исправлений: проверяем известные значения и честно учитываем потери.
print("Повторных номеров в рабочей подвыборке:", toy_ready["PassengerId"].duplicated().sum())
print("Известные Age в допустимых пределах:", toy_ready["Age"].dropna().between(0, 120).all())
print("Известные Fare неотрицательны:", (toy_ready["Fare"].dropna() >= 0).all())
print("Известные Pclass допустимы:", toy_ready["Pclass"].dropna().isin([1, 2, 3]).all())


,PassengerId,Age,Sex,Pclass,Fare,Embarked
0,False,False,False,False,False,False
1,False,True,False,False,False,False
2,False,True,False,True,False,True
3,False,True,False,False,True,False
4,False,True,False,False,False,True
7,False,False,False,False,False,False
8,False,False,False,False,False,True


PassengerId    0
Age            4
Sex            0
Pclass         1
Fare           1
Embarked       3
dtype: int64

Age            4
Embarked       3
Fare           1
Pclass         1
PassengerId    0
Sex            0
dtype: int64

Цепочка дала тот же ответ: True
Повторных номеров в рабочей подвыборке: 0
Известные Age в допустимых пределах: True
Известные Fare неотрицательны: True
Известные Pclass допустимы: True


## Возвращаемся к своему Titanic
Переменные toy, toy_clean и toy_ready относятся только к вымышленному примеру. Ваши пассажиры находятся в df. Не присваивайте df значение toy_ready. Теперь используйте знакомые приёмы самостоятельно.


## Практика P1. Найдите пропуски в своём Titanic

**Для чего:** узнать, для какого вопроса хватает сведений. Для сравнения классов неизвестный возраст сам по себе не мешает.

1. Покажите `df.isna()` для первых пяти строк.
2. Отдельно посчитайте `df["Age"].count()` и `df["Age"].isna().sum()`.
3. Вычислите долю пропусков Age: количество неизвестных возрастов / число всех пассажиров × 100.
4. Покажите `df.isna().sum().sort_values(ascending=False)` и прочитайте результат.

**Ориентиры:** 714 известных возрастов, 177 неизвестных; 687 пропусков Cabin и 2 Embarked. Для Age известные + неизвестные = 891. Отдельный файл отчёта создавать не нужно.


In [9]:
# P1: запишите здесь свои команды по шагам.
pass


### Вопрос и собственный вывод

Почему отсутствие Age не мешает вопросу о доле выживших по Pclass? Назовите нужные поля и знаменатель.

? ВЫВОД: объясните своими словами, назовите числа и ограничение.


## Практика P2. Отличите подозрение от доказанной ошибки

**Для чего:** не удалить настоящих пассажиров по совпавшему билету или необычной цене.

Посчитайте полные повторы строк, повторные PassengerId, повторные вхождения Ticket. Отдельно покажите всех владельцев повторяющегося Ticket через `duplicated(keep=False)`. Проверьте списки Pclass `[1, 2, 3]`, Sex `["female", "male"]`, Survived `[0, 1]`, отрицательные Age/Fare и нулевой Fare.

**Ориентиры:** полных повторов и повторов PassengerId нет; 210 повторных вхождений Ticket, отрицательных Age/Fare нет, 15 нулевых Fare. 210 — не число уникальных общих билетов и не число всех их владельцев.


In [10]:
# P2: запишите здесь свои команды по шагам.
pass


### Вопрос и собственный вывод

Почему нельзя удалить пассажиров только по повторяющемуся Ticket или Fare=0?

? ВЫВОД: объясните своими словами, назовите числа и ограничение.


## Практика P3. Сохраните неизвестные факты и добавьте понятные отметки

**Для чего:** подготовить удобную таблицу для следующих сравнений, не выдавая предположение за исторический факт.

`clean = df.copy()` создаёт рабочую копию в памяти; файл на диске ещё не появляется. Добавьте `Age_known` через `.notna()` и `Embarked_for_analysis` через `.fillna("Неизвестно")`. `fillna` меняет только пропуски; `.notna()` отвечает, известно ли значение.

Покажите два пассажирских исходных Embarked с пропусками рядом с новой подписью. Проверьте, что исходные Age и Embarked сохранили 177 и 2 пропуска. Исходный Embarked не перезаписывайте.


In [11]:
# P3: запишите здесь свои команды по шагам.
pass


### Вопрос и собственный вывод

Стала ли после fillna известна информация о порте? Зачем сохранять исходный столбец?

? ВЫВОД: объясните своими словами, назовите числа и ограничение.


## Практика P4. Проверьте, как удаление строк меняет ответ

**Для чего:** увидеть, что уменьшение таблицы не означает улучшение исследования.

Создайте `age_only = clean.dropna(subset=["Age"])` и `complete_rows = clean.dropna()`. `subset` перечисляет только поля, по которым проверяем пропуски. Без subset проверяются все поля.

Для clean и двух подвыборок отдельно выведите число строк, число выживших и долю выживших. `.mean()` у Survived с числами 0/1 даёт долю единиц; умножение на 100 переводит её в проценты. Всегда называйте знаменатель.

Ожидаются 891, 714 и 183 строки. Основной результат сохраняет всех 891 пассажира: `stage_result = clean.copy()`. Это имя означает готовую таблицу для ячейки сохранения.


In [12]:
# P4: запишите здесь свои команды по шагам.
pass


### Вопрос и собственный вывод

Почему доля 67,21% после полного dropna не доказывает улучшение данных? Кого мы потеряли?

? ВЫВОД: объясните своими словами, назовите числа и ограничение.


## Как сохранить CSV: index=False на маленьком примере

Индекс — номера строк слева в выводе pandas. Он не равен PassengerId. Если сохранить `index=True`, индекс запишется как дополнительный первый столбец. Для нашей пассажирской таблицы используем `index=False`.

В примере ниже `StringIO` — файл в памяти только для показа. Обе версии выводятся на экран и читаются обратно. В сдачу эти демонстрационные файлы не входят. `.to_csv(...)` сохраняет данные, `.read_csv(...)` читает; `print` показывает именно текст CSV.


In [13]:
from io import StringIO
save_example = pd.DataFrame({"PassengerId": [10, 20], "Age": [18, 30]})
csv_with_index = save_example.to_csv(index=True)
csv_without_index = save_example.to_csv(index=False)
print("CSV с техническим индексом:\n" + csv_with_index)
print("CSV без технического индекса:\n" + csv_without_index)
display(pd.read_csv(StringIO(csv_with_index)))
display(pd.read_csv(StringIO(csv_without_index)))


,Unnamed: 0,PassengerId,Age
0,0,10,18
1,1,20,30


,PassengerId,Age
0,10,18
1,20,30


CSV с техническим индексом:
,PassengerId,Age
0,10,18
1,20,30

CSV без технического индекса:
PassengerId,Age
10,18
20,30



## Домашняя работа
H1–H3 продолжают тот же файл. Ориентир 45–60 минут. Копирование примеров без расчётов на Titanic и собственных объяснений не заменяет выполненную работу.


### H1. Измените пример и испытайте проверку

Создайте отдельную копию вымышленной таблицы. Измените одно значение на допустимое, другое — на недопустимое; заранее напишите ожидаемый ответ. Запустите подходящую проверку, покажите найденные строки и объясните результат. Titanic не меняйте. Можно выбрать Age, Fare или Pclass. Напишите, почему отсутствие ошибок в исходном Titanic само по себе не доказывает правильность вашей проверки.


In [14]:
# H1: выполните задание и сохраните результаты.
pass


### Собственное объяснение

? ВЫВОД: ответьте своими словами по пунктам задания.


### H2. Объясните решения по своим данным

Прямо в Markdown ноутбука заполните пять пунктов: Age, Cabin, Embarked, Ticket, Fare. Для каждого: что заметили → сколько записей затронуто → что решили → почему это подходит вопросу. Числа подтвердите кодом. Отдельный CSV или README с журналом не нужен. Уточните смысл числа повторов Ticket и отличие нуля от пропуска.


In [15]:
# H2: выполните задание и сохраните результаты.
pass


### Собственное объяснение

? ВЫВОД: ответьте своими словами по пунктам задания.


### H3. Сохраните и проверьте свой результат

Убедитесь, что stage_result содержит основной clean, и выполните готовую ячейку сохранения ниже. Она создаст только `data/titanic_clean.csv`, откроет его обратно и сверит значения. В Markdown объясните: почему осталось 891 пассажир; почему в пригодной для анализа таблице всё ещё есть пропуски; как проверили сохранность PassengerId и Survived. Перед сдачей сохраните результаты выполнения ноутбука.


In [16]:
# H3: выполните задание и сохраните результаты.
pass


### Собственное объяснение

? ВЫВОД: ответьте своими словами по пунктам задания.


## Готовая ячейка сохранения
Она сохраняет только итоговый пассажирский CSV. Все пояснения остаются в ноутбуке.


In [17]:
from pathlib import Path
if not isinstance(stage_result, pd.DataFrame):
    print("Сохранение пока не выполнено. Завершите P1–P4 и укажите stage_result = clean.copy().")
else:
    # В основном Titanic сохраняем исходные сведения. Опыт с удалением — в отдельных копиях.
    pd.testing.assert_frame_equal(
        stage_result[SOURCE_COLUMNS].sort_values("PassengerId").reset_index(drop=True),
        raw.sort_values("PassengerId").reset_index(drop=True), check_dtype=False)
    Path("data").mkdir(exist_ok=True)
    result_path = Path("data/titanic_clean.csv")
    stage_result.to_csv(result_path, index=False, encoding="utf-8")
    reopened = pd.read_csv(result_path, encoding="utf-8-sig")
    pd.testing.assert_frame_equal(stage_result.reset_index(drop=True), reopened, check_dtype=False)
    print("Сохранён и открыт обратно:", result_path)
    print("Строк:", len(reopened), "Столбцов:", len(reopened.columns))
    print("Сдайте только solution.ipynb и data/titanic_clean.csv. Пояснения — в Markdown ноутбука.")


Сохранение пока не выполнено. Завершите P1–P4 и укажите stage_result = clean.copy().


## Какие ещё способы очистки существуют — обзор, не новое ДЗ

Не все приёмы нужны каждому набору. Сначала определяем проблему и цель; оригинал сохраняем.

| Приём | Для чего используют | Ограничение |
|---|---|---|
| Заполнение пропусков константой, медианой, по группе | Подготовить данные для конкретного расчёта или модели | Это предположение, а не восстановленный факт; медиану для модели считают по обучающей части |
| Перенос последнего значения, интерполяция | Отдельные задачи с упорядоченными наблюдениями, например измерениями по времени | Порядок пассажиров не делает интерполяцию Age обоснованной |
| Приведение текста к единому виду | Пробелы, регистр, известные варианты одной категории | Не объединять разные по смыслу категории |
| Преобразование типов и дат | Числа, даты и логические значения записаны текстом | После преобразования проверить новые пропуски, формат даты и единицы |
| Удаление полных дублей, разбор конфликтов ID | Повторная загрузка или противоречащие записи | Совпадение одного обычного поля не доказывает дубль человека |
| Проверка диапазонов и выбросов | Найти невозможные или необычные значения | Необычное значение не удаляют автоматически |
| Согласование единиц и форматов | Рубли/копейки, годы/месяцы, десятичные разделители | Сначала установить исходную единицу и правило перевода |
| Исправление по надёжному справочнику | Исправить известную опечатку или код | Нужны источник и объяснимое соответствие |
| Проверка связей между таблицами | После соединения не потерять строки и не размножить записи | Проверять ключи и количество строк до и после; подробнее в последующих темах |

**Дополнительный контроль:** список и порядок столбцов; типы `.dtypes`; число строк `len`; число известных значений `.count()`; пропуски `.isna().sum()`; уникальность ID; допустимые категории `.isin()`; согласованность связанных полей; сравнение до/после; повторное чтение сохранённого файла. Проверки должны находить и заранее заданную ошибку.

Пример согласованности из прошлого урока: `Is_alone` должен совпадать с `Family_size == 1`. Для этого урока достаточно понимать проверку, отдельный файл отчёта не требуется.


## Как сдать без лишних файлов

1. Завершите P1–P4 и H1–H3, заполните собственные выводы.
2. Перезапустите среду и выполните ячейки сверху вниз: Restart session / Restart kernel → Run all / Выполнить всё. В Colab при необходимости снова загрузите исходный CSV.
3. Проверьте сообщение «Сохранён и открыт обратно». Во вкладке **Файлы → data** скачайте `titanic_clean.csv`.
4. Скачайте выполненный `.ipynb` через **Файл → Скачать → Скачать .ipynb** и назовите `solution.ipynb`.
5. В своём GitHub откройте `homework/lesson-09/`, загрузите `solution.ipynb`. Для вложенной папки `data` можно перетащить подготовленную папку с CSV через **Add file → Upload files**. Сохраните изменения кнопкой **Commit changes**.
6. Откройте репозиторий и проверьте оба пути: `homework/lesson-09/solution.ipynb` и `homework/lesson-09/data/titanic_clean.csv`.

Папка data из Colab сама в GitHub не переносится. Исходный titanic.csv нужен для работы, но в составе ответа повторно загружать его не требуется. Старые manifest, отчёты качества и README из прежней версии урока можно не сдавать; их отсутствие не является ошибкой этого задания. Если уже сдали старый расширенный набор, повторная сдача только из-за оформления не требуется.

**Самопроверка:** могу объяснить цепочку методов; отличаю пропуск от нуля; не удалил пассажиров по Ticket; сохранил 891 запись и 342 исхода Survived=1; объяснил отбор и знаменатель; итоговый CSV повторно открывается. Зачёт выставляется по содержанию работы и объяснениям, а не по количеству технических файлов.


## Источники и границы
Учебная копия Titanic из материалов pandas. Это часть пассажирских записей, не все пассажиры и экипаж. Подходы к пропускам и проверкам адаптированы из Pandas Workout, Reuven M. Lerner. Дополнительный справочник преподавателя — Python for Data Science. Учебный пример из 10 строк составлен для курса. Связь в наблюдаемых данных не доказывает причину.
